In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv(r"C:\Users\bonga\OneDrive\Desktop\revenue-leakage-analytic\data\cleaned\order_items.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nSample records:")
print(df.head())


Dataset shape: (174849, 10)

Columns:
['order_id', 'product_id', 'quantity', 'unit_price', 'discount_pct', 'net_sales', 'estimated_cost', 'gross_profit', 'profit_margin_pct', 'gross_sales']

Missing values:
order_id             0
product_id           0
quantity             0
unit_price           0
discount_pct         0
net_sales            0
estimated_cost       0
gross_profit         0
profit_margin_pct    0
gross_sales          0
dtype: int64

Sample records:
   order_id product_id  quantity  unit_price  discount_pct  net_sales  \
0  O0000001     P00626         2    14832.33          7.15   27543.64   
1  O0000002     P00091         1    18908.22          3.15   18312.61   
2  O0000003     P00014         1     1406.92          3.54    1357.12   
3  O0000004     P00766         1    11701.42          7.00   10882.32   
4  O0000004     P00634         1     5347.07         11.52    4731.09   

   estimated_cost  gross_profit  profit_margin_pct  gross_sales  
0        23557.62       3986

In [3]:

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [4]:
features = [
    "discount_pct",
    "net_sales",
    "estimated_cost",
    "gross_profit",
    "profit_margin_pct"
]


In [5]:
# Keep only available features
features = [col for col in features if col in df.columns]
X = df[features].replace([np.inf, -np.inf], np.nan)
X.sample(4)

,discount_pct,net_sales,estimated_cost,gross_profit,profit_margin_pct
151603,14.17,12164.49,9040.94,3123.55,25.677608
1444,6.34,8131.90,8159.08,-27.18,-0.334239
76045,1.00,3267.95,2450.77,817.18,25.005891
134001,9.16,2593.79,2028.25,565.54,21.803616


In [6]:
# Remove rows where every selected feature is missing
valid_rows = X.notna().any(axis=1)
valid_rows.sample(3)

52650    True
27233    True
24688    True
dtype: bool

In [7]:
X = X.loc[valid_rows].copy()

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    IsolationForest(
        n_estimators=200,
        contamination=0.02,
        random_state=42
    )
)

In [8]:
model.fit(X)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('simpleimputer', ...), ('standardscaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite

In [9]:
df.loc[X.index, "anomaly_flag"] = (model.predict(X) == -1).astype(int)

df.loc[X.index, "anomaly_score"] = (-model.decision_function(X))

In [10]:
df["anomaly_flag"] = df["anomaly_flag"].fillna(0).astype(int)
df.sample(2)

,order_id,product_id,quantity,unit_price,discount_pct,net_sales,estimated_cost,gross_profit,profit_margin_pct,gross_sales,anomaly_flag,anomaly_score
166899,O0095543,P00695,1,7415.67,3.40,7163.54,5361.04,1802.50,25.162141,7415.67,0,-0.264127
102576,O0058753,P00444,1,11871.79,5.95,11165.42,11003.04,162.38,1.454312,11871.79,0,-0.261401


In [11]:
total = len(df)
flagged = int((df["anomaly_flag"] == 1).sum())
print("Total records:", total)
print("Flagged anomalies:", flagged)
print("Anomaly rate:", round(flagged / total * 100, 2), "%")


Total records: 174849
Flagged anomalies: 3497
Anomaly rate: 2.0 %


In [12]:
comparison = df.groupby("anomaly_flag").agg(
    transaction_count=("order_id", "count"),
    avg_discount=("discount_pct", "mean"),
    avg_profit_margin=("profit_margin_pct", "mean"),
    negative_profit_count=(
        "gross_profit",
        lambda x: (x < 0).sum()
    ),
    negative_profit_rate=(
        "gross_profit",
        lambda x: (x < 0).mean() * 100
    )
).round(2)

print(comparison)


              transaction_count  avg_discount  avg_profit_margin  \
anomaly_flag                                                       
0                        171352          8.28              14.80   
1                          3497         40.34             -49.11   

              negative_profit_count  negative_profit_rate  
anomaly_flag                                               
0                             20425                 11.92  
1                              2450                 70.06  


In [13]:
flagged = df[df["anomaly_flag"] == 1]
print("Flagged line items:", len(flagged))
print("Unique affected orders:", flagged["order_id"].nunique())
print( "Percentage of all orders affected:",round(flagged["order_id"].nunique()/ df["order_id"].nunique() * 100,2),"%")


Flagged line items: 3497
Unique affected orders: 3451
Percentage of all orders affected: 3.45 %


In [14]:
df.to_csv("anomaly_detection_results.csv", index=False)

In [15]:
# Select anomalous line items
anomalies = df[df["anomaly_flag"] == 1].copy()

In [16]:
# Assign investigation reasons using business rules
conditions = [
    (anomalies["discount_pct"] > 40) &
    (anomalies["profit_margin_pct"] < 5),

    anomalies["gross_profit"] < 0,

    anomalies["discount_pct"] > 40
]

In [17]:
reasons = [
    "High discount with low margin",
    "Negative gross profit - validate costs",
    "High discount - review pricing"
]


In [18]:
anomalies["investigation_reason"] = np.select(
    conditions,
    reasons,
    default="Unusual transaction pattern - investigate"
)


In [19]:
# Sort the most unusual records first
anomalies = anomalies.sort_values(
    "anomaly_score",
    ascending=False
)

In [20]:
# Keep useful columns that exist in your dataset
report_columns = [
    "order_id",
    "product_id",
    "discount_pct",
    "net_sales",
    "estimated_cost",
    "gross_profit",
    "profit_margin_pct",
    "anomaly_score",
    "investigation_reason"
]

In [21]:
report_columns = [
    c for c in report_columns if c in anomalies.columns
]

report = anomalies[report_columns]

In [22]:
report.to_csv("anomaly_investigation_report.csv", index=False)
print(report.sample(10).to_string(index=False))
print("\nReport saved successfully.")

order_id product_id  discount_pct  net_sales  estimated_cost  gross_profit  profit_margin_pct  anomaly_score                      investigation_reason
O0021071     P00419          7.14   62874.34        42977.82      19896.52          31.644897       0.068980 Unusual transaction pattern - investigate
O0004879     P00632         52.09    1233.26         2006.92       -773.66         -62.732919       0.001375             High discount with low margin
O0081505     P00122         64.82    2201.14         3727.41      -1526.27         -69.339978       0.024272             High discount with low margin
O0041370     P00117         37.37   19967.00        28233.46      -8266.46         -41.400611       0.020219    Negative gross profit - validate costs
O0061623     P00342         62.40   11372.60        20058.60      -8686.00         -76.376554       0.074214             High discount with low margin
O0004267     P00735         12.72   51067.39        38553.39      12514.00          24.504875 

In [23]:
print("Flagged transactions:", int(df["anomaly_flag"].sum()))
print(df.loc[df["anomaly_flag"] == 1].head(10))

Flagged transactions: 3497
     order_id product_id  quantity  unit_price  discount_pct  net_sales  \
35   O0000025     P00404         1     5495.54         55.61    2439.47   
111  O0000070     P00791         1     4648.41         58.41    1933.27   
203  O0000123     P00747         1    16936.36         57.91    7128.51   
250  O0000153     P00179         3     1450.65         60.12    1735.56   
292  O0000174     P00094         1    17726.97         58.15    7418.74   
473  O0000282     P00138         3    20364.75          9.59   55235.31   
474  O0000283     P00428         2    17795.10         47.25   18773.83   
481  O0000288     P00672         2      878.88         60.91     687.11   
578  O0000344     P00770         2    13449.36         60.65   10584.65   
639  O0000376     P00106         1    14881.64         54.68    6744.36   

     estimated_cost  gross_profit  profit_margin_pct  gross_sales  \
35          4814.22      -2374.75         -97.346965      5495.54   
111      